In [123]:
import json
import random

from datasets import Dataset
from sklearn.model_selection import train_test_split

from transformers import AutoTokenizer

In [ ]:
# load generated data
with open("dataset_positive.json", "r", encoding="utf-8") as f:
    positive_data = json.load(f)

with open("dataset_negative.json", "r", encoding="utf-8") as f:
    negative_data = json.load(f)

print(len(positive_data))
print(len(negative_data))

1192
1392


In [125]:
# convert character annotations to BIO labels

LABELS = [
    "O",
    "B-MOUNTAIN",
    "I-MOUNTAIN"
]

label2id = {label: i for i, label in enumerate(LABELS)}
id2label = {i: label for label, i in label2id.items()}


def convert_to_bio(example):

    text = example["text"]
    tokens = text.split()

    labels = ["O"] * len(tokens)

    for entity in example["entities"]:

        start = entity["start"]
        end = entity["end"]

        current = 0

        for i, token in enumerate(tokens):

            token_start = current
            token_end = current + len(token)

            if token_end > start and token_start < end:

                if token_start == start:
                    labels[i] = "B-MOUNTAIN"
                else:
                    labels[i] = "I-MOUNTAIN"

            current = token_end + 1

    return {
        "tokens": tokens,
        "ner_tags": [label2id[label] for label in labels]
    }


dataset = []

for sample in positive_data:
    dataset.append(convert_to_bio(sample))

for sample in negative_data:
    dataset.append(convert_to_bio(sample))

random.shuffle(dataset)

print(len(dataset))

2584


In [126]:
# split dataset

train_data, valid_data = train_test_split(
    dataset,
    test_size=0.1,
    random_state=42,
    shuffle=True
)

train_dataset = Dataset.from_list(train_data)
valid_dataset = Dataset.from_list(valid_data)

print(len(train_dataset))
print(len(valid_dataset))

2325
259


In [127]:
# tokenize and align labels

MODEL_NAME = "bert-base-cased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def tokenize_and_align_labels(examples):

    tokenized = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        max_length=256
    )

    labels = []

    for i in range(len(examples["tokens"])):

        word_ids = tokenized.word_ids(batch_index=i)

        previous_word = None
        label_ids = []

        for word_idx in word_ids:

            if word_idx is None:
                label_ids.append(-100)

            elif word_idx != previous_word:
                label_ids.append(examples["ner_tags"][i][word_idx])

            else:
                current = examples["ner_tags"][i][word_idx]

                if current == label2id["B-MOUNTAIN"]:
                    current = label2id["I-MOUNTAIN"]

                label_ids.append(current)

            previous_word = word_idx

        labels.append(label_ids)

    tokenized["labels"] = labels

    return tokenized


In [128]:
MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

label2id = {
    "O": 0,
    "B-MOUNTAIN": 1,
    "I-MOUNTAIN": 2
}

id2label = {v: k for k, v in label2id.items()}

In [129]:
def tokenize_and_align_labels(examples):
    tokenized = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        max_length=128
    )

    batch_labels = []

    for i, tags in enumerate(examples["ner_tags"]):

        word_ids = tokenized.word_ids(batch_index=i)

        previous_word = None
        labels = []

        for word_idx in word_ids:

            if word_idx is None:
                labels.append(-100)

            elif word_idx != previous_word:
                labels.append(tags[word_idx])

            else:
                current = tags[word_idx]

                if current == 1:          # B-MOUNTAIN
                    current = 2           # I-MOUNTAIN

                labels.append(current)

            previous_word = word_idx

        batch_labels.append(labels)

    tokenized["labels"] = batch_labels

    return tokenized

In [130]:
train_dataset = train_dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=["tokens", "ner_tags"] # видаляємо старі колонки, щоб Trainer не сварився
)

valid_dataset = valid_dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=["tokens", "ner_tags"]
)

Map: 100%|██████████| 259/259 [00:00<00:00, 5220.00 examples/s]


In [131]:
from datasets import Dataset

for i in range(5):
    print(dataset[i]["tokens"])
    print([id2label[x] for x in dataset[i]["ner_tags"]])
    print()
    
hf_dataset = Dataset.from_list(dataset)

tokenized_dataset = hf_dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=hf_dataset.column_names
)
sample = tokenized_dataset[2]

tokens = tokenizer.convert_ids_to_tokens(sample["input_ids"])

for t, l in zip(tokens, sample["labels"]):
    print(t, l)

['Meltwater', 'runoff', 'from', 'the', 'extensive', 'snowpack', 'on', 'Ancohuma', 'feeds', 'several', 'crucial', 'lowland', 'river', 'networks.']
['O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-MOUNTAIN', 'O', 'O', 'O', 'O', 'O', 'O']

['Meltwater', 'runoff', 'from', 'the', 'extensive', 'snowpack', 'on', 'Lyskamm', 'feeds', 'several', 'crucial', 'lowland', 'river', 'networks.']
['O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-MOUNTAIN', 'O', 'O', 'O', 'O', 'O', 'O']

["Reventador's", 'algorithmic', 'trading', 'bot', 'executed', 'a', 'bad', 'trade', 'due', 'to', 'a', 'delayed', 'market', 'data', 'feed.']
['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']

['In', 'chapter', 'four', 'of', 'the', 'biography,', 'Tunari', 'is', 'described', 'as', 'a', 'tiny', 'garage', 'project', 'that', 'blew', 'up', 'overnight.']
['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']

['Platforms', 'engineered', 'by', 'Fairweather', 'are', 'heavily', 'utilized', 'i

Map: 100%|██████████| 2584/2584 [00:00<00:00, 6195.86 examples/s]

[CLS] -100
rev 0
##ent 0
##ador 0
' 0
s 0
algorithm 0
##ic 0
trading 0
bot 0
executed 0
a 0
bad 0
trade 0
due 0
to 0
a 0
delayed 0
market 0
data 0
feed 0
. 0
[SEP] -100


In [132]:
print(label2id)
print(id2label)

{'O': 0, 'B-MOUNTAIN': 1, 'I-MOUNTAIN': 2}
{0: 'O', 1: 'B-MOUNTAIN', 2: 'I-MOUNTAIN'}


In [133]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 2470.94it/s]
[transformers] DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [134]:
import evaluate
import numpy as np

seqeval = evaluate.load("seqeval")

In [135]:
def compute_metrics(eval_pred):

    predictions, labels = eval_pred

    predictions = np.argmax(predictions, axis=-1)

    true_predictions = []
    true_labels = []

    for prediction, label in zip(predictions, labels):

        pred = []
        gold = []

        for p, l in zip(prediction, label):

            if l == -100:
                continue

            pred.append(id2label[p])
            gold.append(id2label[l])

        true_predictions.append(pred)
        true_labels.append(gold)

    return seqeval.compute(
        predictions=true_predictions,
        references=true_labels
    )

In [136]:
from transformers import TrainingArguments
from transformers import DataCollatorForTokenClassification
from transformers import Trainer

In [137]:
training_args = TrainingArguments(
    output_dir="./mountain_ner",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="overall_f1",
    greater_is_better=True
)

In [138]:
data_collator = DataCollatorForTokenClassification(tokenizer)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)



In [139]:
trainer.train()

Epoch,Training Loss,Validation Loss,Mountain,Overall Precision,Overall Recall,Overall F1,Overall Accuracy
1,0.036372,0.015577,"{'precision': 0.975, 'recall': 0.9140625, 'f1': 0.9435483870967741, 'number': 128}",0.975000,0.914062,0.943548,0.996426
2,0.009468,0.010662,"{'precision': 0.9836065573770492, 'recall': 0.9375, 'f1': 0.96, 'number': 128}",0.983607,0.937500,0.960000,0.997267
3,0.003749,0.009666,"{'precision': 0.9838709677419355, 'recall': 0.953125, 'f1': 0.9682539682539683, 'number': 128}",0.983871,0.953125,0.968254,0.997687
4,0.001977,0.009267,"{'precision': 0.9838709677419355, 'recall': 0.953125, 'f1': 0.9682539682539683, 'number': 128}",0.983871,0.953125,0.968254,0.997687
5,0.000805,0.010401,"{'precision': 0.983739837398374, 'recall': 0.9453125, 'f1': 0.9641434262948206, 'number': 128}",0.983740,0.945312,0.964143,0.997477


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  6.24it/s]
c:\Users\Andrew\Documents\projects\test_assignment\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  6.53it/s]
c:\Users\Andrew\Documents\projects\test_assignment\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.25it/s]
c:\Users\Andrew\Documents\projects\test_assignment\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 

TrainOutput(global_step=730, training_loss=0.02584245514175663, metrics={'train_runtime': 776.604, 'train_samples_per_second': 14.969, 'train_steps_per_second': 0.94, 'total_flos': 75483131164200.0, 'train_loss': 0.02584245514175663, 'epoch': 5.0})

In [140]:
print(model.training)
print(trainer.model.training)

False
False


In [141]:
import torch
from transformers import pipeline

# беремо саме модель з Trainer
trainer.model.eval()

ner = pipeline(
    "ner",
    model=trainer.model,
    tokenizer=tokenizer,
    aggregation_strategy="simple"
)

tests = [
    "The climbers reached the summit of Everest at dawn.",
    "K2 is considered one of the most dangerous mountains in the world.",
    "Meltwater from Lanín feeds several river networks below.",
    "Local indigenous communities hold deep cultural reverence for Wollumbin / Mount Warning.",
    "Mount Everest is the highest mountain on Earth.",
    "I just updated my Ushba app and it keeps crashing.",
    "Can someone explain why Nun keeps throwing a null pointer exception?",
    "According to the documentation, Cerro Plata requires 16GB of RAM."
]

for text in tests:
    print("=" * 80)
    print(text)

    results = ner(text)

    if len(results) == 0:
        print("FOUND: None")
    else:
        for r in results:
            print(
                f"{r['word']:<35} "
                f"{r['entity_group']:<10} "
                f"score={r['score']:.3f}"
            )

The climbers reached the summit of Everest at dawn.
everest                             MOUNTAIN   score=0.995
K2 is considered one of the most dangerous mountains in the world.
k2                                  MOUNTAIN   score=0.997
mountains                           MOUNTAIN   score=0.980
Meltwater from Lanín feeds several river networks below.
lanin                               MOUNTAIN   score=0.999
Local indigenous communities hold deep cultural reverence for Wollumbin / Mount Warning.
wollumbin / mount warning.          MOUNTAIN   score=0.998
Mount Everest is the highest mountain on Earth.
mount everest                       MOUNTAIN   score=0.968
mountain                            MOUNTAIN   score=0.992
I just updated my Ushba app and it keeps crashing.
FOUND: None
Can someone explain why Nun keeps throwing a null pointer exception?
FOUND: None
According to the documentation, Cerro Plata requires 16GB of RAM.
FOUND: None


In [142]:
trainer.save_model("./mountain_ner_model")
tokenizer.save_pretrained("./mountain_ner_model")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.29it/s]


('./mountain_ner_model\\tokenizer_config.json',
 './mountain_ner_model\\tokenizer.json')

In [143]:
trainer.model.eval()

DistilBertForTokenClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
          (ffn): FFN(
            (dropout): Dropout(p=0

In [ ]:
from transformers import AutoModelForTokenClassification, AutoTokenizer, pipeline


model = trainer.model
ner_pipeline = pipeline(
    "ner",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="simple"  # об'єднує B- / I- токени в одну сутність
)

tests = [
    "Let's organize the purchase of tickets for our upcoming hike to Mount Hoverla, considering that we need to reserve two seats in first class and prepare confirmation letters for all group members.",
"My dream is to visit the summit of Everest.",
"Someone please investigate why the NullPointerException in the authentication module keeps throwing the server into state 500, and provide a short report with suggestions for code refactoring.",
"We have located our base camp directly at the foot of Mont Blanc, closer to the eastern slope, where tiny storage facilities for supplies and a first aid station are already prepared.",
"The technical documentation for the Cerro Plata project indicates that the minimum server configuration should include 16GB of RAM, a processor with a frequency of at least 2.8GHz and an SSD of at least 512GB for uninterrupted data processing."
"Let's organize the purchase of tickets for our upcoming hike to Mount Hoverla, considering that we need to reserve two first-class seats and prepare confirmation letters for all group members.",
"The latest patch for the Ushba mobile application fixes a critical vulnerability that caused the application to randomly crash when trying to synchronize GPS data with the server.",
"Someone please investigate why the NullPointerException in the authentication module keeps throwing the server into state 500, and provide a short report with suggestions for code refactoring.",
"We have located our base camp directly at the foot of Mont Blanc, closer to the eastern slope, where tiny storage facilities for supplies and a first aid station are already prepared.",
"The technical documentation for the Cerro Plata project indicates that the minimum server configuration should include 16GB of RAM, a processor with with a frequency of at least 2.8 GHz and an SSD disk with a capacity of at least 512 GB for uninterrupted data processing."
"Let's organize the purchase of tickets for our upcoming hike to Mount Hoverla, considering that we need to reserve two first-class seats and prepare confirmation letters for all group members.",
"The latest patch for the Ushba mobile application fixes a critical vulnerability that caused the application to randomly crash when trying to synchronize GPS data with the server.",
"Someone please investigate why the NullPointerException in the authentication module keeps throwing the server into state 500, and provide a short report with suggestions for code refactoring.",
"We have located our base camp directly at the foot of Mont Blanc, closer to the eastern slope, where tiny storage facilities for supplies and a first aid station are already prepared.",
"The technical documentation for the Cerro Plata project indicates that the minimum server configuration should include 16GB of RAM, a processor with with a frequency of at least 2.8 GHz and an SSD disk with a capacity of at least 512 GB for uninterrupted data processing."
]

for text in tests:
    results = ner_pipeline(text)
    mountains = [r["word"] for r in results if r["entity_group"] == "MOUNTAIN"]
    print(f"TEXT:  {text}")
    print(f"FOUND: {mountains if mountains else '—'}")
    print()

TEXT:  Let's organize the purchase of tickets for our upcoming hike to Mount Hoverla, considering that we need to reserve two seats in first class and prepare confirmation letters for all group members.
FOUND: ['mount hoverla']

TEXT:  My dream is to visit the of Everest.
FOUND: —

TEXT:  Someone please investigate why the NullPointerException in the authentication module keeps throwing the server into state 500, and provide a short report with suggestions for code refactoring.
FOUND: —

TEXT:  We have located our base camp directly at the foot of Mont Blanc, closer to the eastern slope, where tiny storage facilities for supplies and a first aid station are already prepared.
FOUND: ['mont blanc']

TEXT:  The technical documentation for the Cerro Plata project indicates that the minimum server configuration should include 16GB of RAM, a processor with a frequency of at least 2.8GHz and an SSD of at least 512GB for uninterrupted data processing.Let's organize the purchase of tickets for 